# PyTorch Schedule Extraction
Model the GPT-3 workload operations using PyTorch and extract execution schedule.

In [1]:
import torch
import torch.nn as nn
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
class OperationTracker:
    """Track operations as they execute in PyTorch computational graph."""

    def __init__(self):
        self.operations = []
        self.op_counter = 0

    def register_operation(self, name, op_type, inputs, output):
        """Register an operation in the schedule."""
        op_info = {
            'id': self.op_counter,
            'name': name,
            'type': op_type,
            'inputs': inputs,
            'output_shape': tuple(output.shape) if hasattr(output, 'shape') else None,
        }
        self.operations.append(op_info)
        self.op_counter += 1
        return self.op_counter - 1

In [3]:
class GPT3Workload(nn.Module):
    """
    PyTorch model implementing the GPT-3 workload from gpt3_175B.yaml

    Workload operations (from YAML):
    1. I[b, m, d] = I_in[b, m, d]                        # Copy
    2. V[b, m, h, e] = I[b, m, d] * WV[h, e, d]          # Value projection
    3. K[b, m, h, e] = I[b, m, d] * WK[h, e, d]          # Key projection
    4. Q[b, m, h, e] = I[b, m, d] * WQ[h, e, d]          # Query projection
    5. QK[b, m, p, h] = Q[b, m, h, e] * K[b, p, h, e]    # Attention scores
    6. QK_softmax[b, m, p, h] = QK[b, m, p, h]           # Softmax
    7. AV[b, m, h, f] = QK_softmax * V[b, p, h, f]       # Attention output
    8. Z[b, m, g] = AV[b, m, h, f] * WZ[h, f, g]         # Output projection
    9. FFA[b, m, c] = Z[b, m, g] * WFFA[g, c]            # Feed-forward A
    10. FFB[b, m, j] = FFA[b, m, c] * WFFB[c, j]         # Feed-forward B
    """

    def __init__(self, B=1, M=8192, H=96, E=128):
        super().__init__()

        # Dimensions from gpt3_175B.yaml
        self.B = B          # Batch size
        self.M = M          # Number of tokens
        self.P = M          # Number of tokens (for attention)
        self.H = H          # Number of heads
        self.E = E          # Embedding dim per head
        self.F = E          # Same as E
        self.D = H * E      # Total embedding dimension (96 * 128 = 12288)
        self.C = 4 * H * E  # Feed-forward dimension (4 * 96 * 128 = 49152)
        self.J = H * E      # Output dimension
        self.G = H * E      # Intermediate dimension

        # Initialize weight matrices (persistent tensors from YAML)
        self.WV = nn.Parameter(torch.randn(H, E, self.D))
        self.WK = nn.Parameter(torch.randn(H, E, self.D))
        self.WQ = nn.Parameter(torch.randn(H, E, self.D))
        self.WZ = nn.Parameter(torch.randn(H, self.F, self.G))
        self.WFFA = nn.Parameter(torch.randn(self.G, self.C))
        self.WFFB = nn.Parameter(torch.randn(self.C, self.J))

        # Operation tracker
        self.tracker = OperationTracker()

    def forward(self, I_in):
        """
        Forward pass implementing all operations from gpt3_175B.yaml

        Args:
            I_in: Input tensor [B, M, D]

        Returns:
            FFB: Final output [B, M, J]
        """

        # Operation 1: I[b, m, d] = I_in[b, m, d] (copy operation)
        I = I_in.clone()
        self.tracker.register_operation('I', 'copy', ['I_in'], I)

        # Operation 2: V[b, m, h, e] = I[b, m, d] * WV[h, e, d]
        V = torch.einsum('bmd,hed->bmhe', I, self.WV)
        self.tracker.register_operation('V', 'matmul', ['I', 'WV'], V)

        # Operation 3: K[b, m, h, e] = I[b, m, d] * WK[h, e, d]
        K = torch.einsum('bmd,hed->bmhe', I, self.WK)
        self.tracker.register_operation('K', 'matmul', ['I', 'WK'], K)

        # Operation 4: Q[b, m, h, e] = I[b, m, d] * WQ[h, e, d]
        Q = torch.einsum('bmd,hed->bmhe', I, self.WQ)
        self.tracker.register_operation('Q', 'matmul', ['I', 'WQ'], Q)

        # Operation 5: QK[b, m, p, h] = Q[b, m, h, e] * K[b, p, h, e]
        QK = torch.einsum('bmhe,bphe->bmph', Q, K)
        self.tracker.register_operation('QK', 'matmul', ['Q', 'K'], QK)

        # Operation 6: QK_softmax[b, m, p, h] = softmax(QK)
        QK_softmax = torch.softmax(QK, dim=2)
        self.tracker.register_operation('QK_softmax', 'softmax', ['QK'], QK_softmax)

        # Operation 7: AV[b, m, h, f] = QK_softmax * V
        AV = torch.einsum('bmph,bphe->bmhe', QK_softmax, V)
        self.tracker.register_operation('AV', 'matmul', ['QK_softmax', 'V'], AV)

        # Operation 8: Z[b, m, g] = AV * WZ
        Z = torch.einsum('bmhf,hfg->bmg', AV, self.WZ)
        self.tracker.register_operation('Z', 'matmul', ['AV', 'WZ'], Z)

        # Operation 9: FFA[b, m, c] = Z * WFFA
        FFA = torch.einsum('bmg,gc->bmc', Z, self.WFFA)
        self.tracker.register_operation('FFA', 'matmul', ['Z', 'WFFA'], FFA)

        # Operation 10: FFB[b, m, j] = FFA * WFFB
        FFB = torch.einsum('bmc,cj->bmj', FFA, self.WFFB)
        self.tracker.register_operation('FFB', 'matmul', ['FFA', 'WFFB'], FFB)

        return FFB

In [4]:
def build_dependency_graph(tracker):
    """
    Build a dependency graph from tracked operations.

    Returns:
        dict: operation_name -> list of dependencies
    """
    dependencies = defaultdict(list)
    op_outputs = {}

    for op in tracker.operations:
        name = op['name']
        inputs = op['inputs']

        # Track what this operation produces
        op_outputs[name] = name

        # Find dependencies
        for inp in inputs:
            if inp in op_outputs and inp != name:
                dependencies[name].append(inp)

    return dependencies

In [5]:
def assign_compute_units(tracker, dependencies, num_units=4):
    """
    Assign operations to compute units based on dependencies.

    Simple scheduling: operations run in parallel if no dependencies
    """
    schedule = []
    unit_end_times = [0] * num_units
    op_end_times = {}

    for op in tracker.operations:
        name = op['name']
        deps = dependencies.get(name, [])

        # Find earliest start time (after all dependencies)
        dep_end_time = max([op_end_times.get(d, 0) for d in deps], default=0)

        # Find first available compute unit
        earliest_unit = min(range(num_units), key=lambda i: unit_end_times[i])
        start_time = max(dep_end_time, unit_end_times[earliest_unit])

        # Assume unit latency
        duration = 1.0
        end_time = start_time + duration

        schedule.append({
            'operation': name,
            'type': op['type'],
            'compute_unit': f'PE_{earliest_unit}',
            'start_cycle': start_time,
            'end_cycle': end_time,
            'duration': duration,
            'dependencies': ', '.join(deps) if deps else 'None'
        })

        unit_end_times[earliest_unit] = end_time
        op_end_times[name] = end_time

    return pd.DataFrame(schedule)



In [6]:
def print_schedule(schedule_df):
    """Print human-readable schedule."""
    print("=" * 110)
    print("GPT-3 WORKLOAD EXECUTION SCHEDULE")
    print("=" * 110)
    print(f"{'Operation':<15} {'Type':<10} {'PE Unit':<12} {'Start':<8} {'End':<8} {'Duration':<10} {'Dependencies':<20}")
    print("-" * 110)

    for _, row in schedule_df.iterrows():
        print(f"{row['operation']:<15} "
              f"{row['type']:<10} "
              f"{row['compute_unit']:<12} "
              f"{row['start_cycle']:<8.1f} "
              f"{row['end_cycle']:<8.1f} "
              f"{row['duration']:<10.1f} "
              f"{row['dependencies']:<20}")

    print("=" * 110)
    print(f"Total execution time: {schedule_df['end_cycle'].max():.1f} cycles")
    print(f"Number of compute units: {len(schedule_df['compute_unit'].unique())}")
    print("=" * 110)

In [7]:
def visualize_schedule(schedule_df, title='GPT-3 Workload Schedule'):
    """Create Gantt chart of execution schedule."""
    fig, ax = plt.subplots(figsize=(14, 8))

    units = sorted(schedule_df['compute_unit'].unique())
    unit_to_y = {unit: i for i, unit in enumerate(units)}

    colors = {
        'copy': 'lightblue',
        'matmul': 'lightgreen',
        'softmax': 'orange'
    }

    for _, row in schedule_df.iterrows():
        y_pos = unit_to_y[row['compute_unit']]
        start = row['start_cycle']
        duration = row['duration']
        color = colors.get(row['type'], 'gray')

        ax.barh(y_pos, duration, left=start, height=0.8,
                color=color, edgecolor='black', linewidth=1)

        ax.text(start + duration/2, y_pos, row['operation'],
                ha='center', va='center', fontsize=9, fontweight='bold')

    ax.set_yticks(range(len(units)))
    ax.set_yticklabels(units)
    ax.set_xlabel('Cycles', fontsize=12, fontweight='bold')
    ax.set_ylabel('Compute Unit (Processing Element)', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, axis='x', alpha=0.3)

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=color, label=op_type, edgecolor='black')
                      for op_type, color in colors.items()]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

    plt.tight_layout()
    return fig


In [ ]:
print("Creating GPT-3 workload model...")
model = GPT3Workload(B=1, M=8192, H=96, E=128)

print("Creating input tensor...")
I_in = torch.randn(1, 8192, 96*128)

print("Running forward pass to track operations...")
with torch.no_grad():
    output = model(I_in)

print(f"✓ Output shape: {output.shape}")
print(f"✓ Tracked {len(model.tracker.operations)} operations\n")


Creating GPT-3 workload model...
Creating input tensor...
Running forward pass to track operations...


In [ ]:
print("Building dependency graph...")
dependencies = build_dependency_graph(model.tracker)

print("\nOperation Dependencies:")
for op, deps in dependencies.items():
    if deps:
        print(f"  {op:15} depends on: {', '.join(deps)}")
    else:
        print(f"  {op:15} has no dependencies")

In [ ]:
print("\n\nAssigning operations to compute units (4 PEs)...")
schedule_df = assign_compute_units(model.tracker, dependencies, num_units=4)

In [ ]:
print_schedule(schedule_df)

In [ ]:
schedule_df.to_csv('gpt3_pytorch_schedule.csv', index=False)
print("\n✓ Schedule saved to gpt3_pytorch_schedule.csv")

In [ ]:
fig = visualize_schedule(schedule_df, 'GPT-3 Workload Execution Schedule (PyTorch Model)')
plt.savefig('gpt3_pytorch_schedule.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved to gpt3_pytorch_schedule.png")
plt.show()

In [ ]:
print("\nSchedule DataFrame:")
schedule_df